In [148]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [149]:
# base imports
import numpy as np 
import pandas as pd
from functools import partial
import pickle
import warnings
import os

# preprocessing
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from bs4 import BeautifulSoup

import re

# NLP packages
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.model_selection import train_test_split
import scipy

from datasets import load_dataset

max_length = 32
# size = 'all'
size = 250
"""
Data Preprocessing Functions
"""
stemmer = PorterStemmer()
lemma = WordNetLemmatizer()

def tokenize(text,vocabulary,max_length = 32):
    # my text was unicode so I had to use the unicode-specific translate function. If your documents are strings, you will need to use a different `translate` function here. `Translated` here just does search-replace. See the trans_table: any matching character in the set is replaced with `None`
    # tokens = [word for word in word_tokenize(text.lower()) if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    tokens = [word for word in text.lower() if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    # tokens = [word for word in word_tokenize(text.lower())]# if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    indexed_tokens = [vocabulary.get(token, -1)+2 for token in tokens]  # 1 for unknown words

    # Trim to 32 tokens or pad with -1s if shorter
    if len(indexed_tokens) < max_length:
        indexed_tokens.extend([0] * (max_length - len(indexed_tokens)))  # Padding with -1
    else:
        indexed_tokens = indexed_tokens[:max_length]  # Trim to n_tokens=max_length

    
    # return [vocabulary.get(token, -1) for token in tokens]  # -1 for unknown words
    return indexed_tokens

def token2index(tokens,vocabulary,max_length = 32):
    # my text was unicode so I had to use the unicode-specific translate function. If your documents are strings, you will need to use a different `translate` function here. `Translated` here just does search-replace. See the trans_table: any matching character in the set is replaced with `None`
    # tokens = [word for word in word_tokenize(text.lower()) if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    
    # tokens = [word for word in word_tokenize(text.lower())]# if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    indexed_tokens = [vocabulary.get(token, -1)+2 for token in tokens]  # 1 for unknown words

    # Trim to 32 tokens or pad with -1s if shorter
    if len(indexed_tokens) < max_length:
        indexed_tokens.extend([0] * (max_length - len(indexed_tokens)))  # Padding with -1
    else:
        indexed_tokens = indexed_tokens[:max_length]  # Trim to n_tokens=max_length

    
    # return [vocabulary.get(token, -1) for token in tokens]  # -1 for unknown words
    return indexed_tokens

def assert_even_length_array(arr):
    if len(arr) % 2 != 0:
        raise AssertionError("Array does not have an even number of elements: {}".format(len(arr)))

def check_order(array):
    prev_entry = None
    ilist = []  
    for i, entry in enumerate(array):
        if entry == prev_entry:
            ilist.append(i)
    
        prev_entry = entry

    return ilist

def sort_data_cfs(array,ilist):
    # given a list of indices, swap the index and the following index?
    for i in ilist:
        temp1 = array[i]
        temp2 = array[i+1]

        array.iloc[i] = temp2
        array.iloc[i+1] = temp1

    return array

def sort_label_cfs(array,ilist):
    # given a list of indices, swap the index and the following index?
    for i in ilist:
        temp1 = array[i]
        temp2 = array[i+1]

        array[i] = temp2
        array[i+1] = temp1

    return array


def get_vector(vec1,vec2):
    
    d_vec = vec2 - vec1
    
    if np.sum(d_vec) == 0:
        
        warnings.warn("A text vector and its' counterfactual vector have the same value. This is probably not right.")
        
    if isinstance(d_vec, scipy.sparse.csr_matrix):  # or any other sparse matrix type
        mag = scipy.linalg.norm(d_vec.todense())

    elif isinstance(d_vec, np.ndarray):
        mag = np.linalg.norm(d_vec)
        # Perform operations specific to dense arrays
    else:
        print("Unknown vector type")
        mag = np.linalg.norm(d_vec)
    
    if mag!=0:
        return d_vec/mag        
    else:
        return np.zeros((np.shape(vec1)[0]))
    # return [d_vec/mag]



## cleaning the text

def cleantext(text):
    # removing the "\"
    text = re.sub("'\''","",text)
    
    # removing special symbols
    text = re.sub("[^ a-zA-Z]","",text)
    
    # removing the whitespaces
    text = ' '.join(text.split())
    
    # convert text to lowercase    
    text = text.lower()
    
    return text

# removing the stopwords
stop_words = set(stopwords.words('english'))
mod_stop_words = set(stopwords.words('english')) - {'not', 'but'}
# stop_words = stop_words - set(['dont','do'])
# def removestopwords(text):
    
#     removedstopword = [word for word in text.split() if word not in mod_stop_words]
#     return ' '.join(removedstopword)

def removestopwords(text):
    
    removedstopword = [word for word in text if word not in mod_stop_words]
    return removedstopword

#Removing the html strips 
def strip_html(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

#Removing the square brackets
def remove_between_square_brackets(text):
    return re.sub(r'\[[^]]*\]', '', text)

#Removing Emails
def remove_Emails(text):
    pattern=r'\S+@\S+'
    text=re.sub(pattern,'',text)
    return text

#Removing URLS
def remove_URLS(text):
    pattern=r'http\S+'
    text=re.sub(pattern,'',text)
    return text

def remove_special_characters(text, remove_digits=True):
    pattern=r'[^a-zA-z0-9\s]'
    text=re.sub(pattern,' ',text)
    return text

#Removing numbers
def remove_numbers(text):
    pattern = r'\d+'
    text = re.sub(pattern, '', text)
    return text

def lematizing(sentence):
    stemSentence = ""
    for word in sentence.split():
        stem = lemma.lemmatize(word)
        stemSentence += stem
        stemSentence += " "
    stemSentence = stemSentence.strip()
    return stemSentence

def stemming(sentence):
    
    stemmed_sentence = ""
    for word in sentence.split():
        stem = stemmer.stem(word)
        stemmed_sentence+=stem
        stemmed_sentence+=" "
        
    stemmed_sentence = stemmed_sentence.strip()
    return stemmed_sentence



def process_data(data):
    # Step 1: clean text (string level)
    data = data.apply(lambda x: strip_html(x))
    data = data.apply(remove_between_square_brackets)
    data = data.apply(lambda x: cleantext(x))
    data = data.apply(remove_special_characters)
    data = data.apply(remove_URLS)
    data = data.apply(remove_Emails)
    data = data.apply(remove_numbers)
    
    # Step 2: tokenize
    data = data.apply(lambda x: word_tokenize(x.lower()))
    
    # Step 3: remove stopwords and stem (token level)
    data = data.apply(lambda tokens: [stemming(token) for token in removestopwords(tokens)])

    return data




def gen_knowledge(data,label,cf=False):
    """
    Takes sequences of pairs of counterfactuals and 
    returns just the originals, with the counterfactual 
    included as an annotation.

    If cf=True, returns the counterfactuals and omits the originals
    """
    
    vectors_in = data['vector']
    text_in = data['text']
    # sparse_array = sp.csr_matrix((0,dlength))

    text_out=[]
    vectors_out = []
    labels = []
    # vectors = np.empty((0,1,dlength))
    cf_text = []
    cf_labels = []
    cf_vectors = []
    
    for i in range(int(np.shape(vectors_in)[0]/2)):
        
        
        if cf:
            vec1,vec2 = vectors_in[i * 2 + 1],vectors_in[i * 2]  # Get the data from the current dataset slice
            label_out = label[i*2+1]
            text = text_in[i * 2 + 1]
            _cf_text=text_in[i * 2]
        else:
            vec1,vec2 = vectors_in[i * 2],vectors_in[i * 2 + 1]  # Get the data from the current dataset slice                
            label_out = label[i*2]
            text = text_in[i * 2]
            _cf_text = text_in[i * 2 + 1]
            
        text_out.append(text)
        labels.append(label_out)
        vectors_out.append(vec1)
    
        cf_vectors.append([vec2])
        cf_labels.append([1 - label_out])
        cf_text.append([_cf_text])
    
    return {'vector':vectors_out,'text':text_out},labels,cf_vectors, cf_labels,cf_text



def compile_K(data,label, paired=False,cf=False,int_out=False):
    
    if paired:
        
        if np.shape(data['vector'])[0]%2 != 0:
            raise ValueError("Can't generate paired counterfactuals with an uneven number of samples")
        
        data,label,vector,labels,cf_text = gen_knowledge(data,label,cf=cf)
        
        # labels = [1 - l for l in label]
        
    else:
        warnings.warn("Non-paired data just creates blanks atm")
        
        vector = [[] for _ in range(np.shape(data['vector'])[0])]
        cf_text = [[] for _ in range(np.shape(data['vector'])[0])]

        labels = [[]]*len(vector)
    
    # labels = np.expand_dims(labels, axis=1)

    magnitude = np.ones(len(vector))
    magnitude = np.expand_dims(magnitude, axis=1)
    

    if int_out:
        for i in range(np.shape(vector)[0]):
            vector[i] = vector[i].astype(int)

    n_samples = len(data['text'])
    print(f'Returning {n_samples} samples with {len(vector)} counterfactuals')
    
    return {'text':data['text'],
            'X':data['vector'],
            'Y':label,
            'K':{'vector':vector,'label':labels,'magnitude':magnitude, 'text':cf_text}}


def compile_ag(X,y,text,cf_X,cf_text):
    
    magnitude = np.ones(len(cf_X))
    magnitude = np.expand_dims(magnitude, axis=1)

    n_samples = len(text)

    print(f'Returning {n_samples} samples with {len(cf_text)} counterfactuals')
    
    return {'text':list(text),
            'X':X,
            'Y':list(y),
            'K':{'X':np.expand_dims(X,axis=1),
                 'Y':np.expand_dims(np.full_like(np.array(y),np.nan),axis=1),
                 'K':np.expand_dims(cf_X,axis=1),
                 'magnitude':np.expand_dims(magnitude,axis=1),
                 'text':np.expand_dims(cf_text,axis=1)}}




In [150]:
ds = load_dataset("SetFit/ag_news")

agnews = pd.read_json('data/original/fizle_AG_News_Llama3-70B-Instruct.json')
agnews_tt = pd.read_json('data/original/counterfactual_ag_news.json')
agnews_all = pd.read_csv('data/original/ag_news.csv')


In [151]:
agnews_tt

,original_idx,input,counterfactual,label,prediction
0,90126,IBM to Quit Making PCs An era will come to an ...,UN to Quit Peacekeeping Operations An era will...,science/technology,world
1,105943,Sun: 1.9 million downloads of Java fix (InfoWo...,UN: 1.9 million affected by drought (InfoWorld...,science/technology,world
2,90395,Former Indonesian intelligence chief says emba...,Indonesian badminton star says his training re...,world,sports
3,68369,RFK due to retain prime soccer seating DC Unit...,RFK due to retain prime embassy seating DC Uni...,sports,world
4,97094,Creditors to forgive up to 80 per cent of Iraq...,Federer to forfeit up to 80 percent of Wimbled...,world,sports
...,...,...,...,...,...
995,11688,"A Gandhi preaches peace in Mideast RAMALLAH, W...",A Gandhi signs multi-million dollar endorsemen...,world,business
996,17020,National League Game Summary - Los Angeles at ...,UN Security Council Condemns Violence in Escal...,sports,world
997,51094,"Cardinals, Red Sox Use Homers to Win Baseball ...",EU and US Impose Sanctions on Russian Oligarch...,sports,world
998,16606,BMC update aims to nip downtime in the bud Bat...,UN update aims to nip conflict in the bud Peac...,science/technology,world


In [152]:
labels = ['world','sports','science/technology','business']
agnews_tt['text_label'] = agnews_tt['label']  

def map_labels_to_ints(labels):
    mapping = {
        "world": 0,
        "sports": 1,
        "business": 2,
        "science/technology": 3
    }
    return [mapping[label] for label in labels]
agnews_tt['prediction'] = map_labels_to_ints(agnews_tt['prediction'].values)
agnews_tt['label'] = map_labels_to_ints(agnews_tt['label'].values)

In [153]:


# X_train, X_test, y_train, y_test = train_test_split(agnews['input'], agnews['label'], test_size=0.2, random_state=42)
# X_test, X_dev, y_test, y_dev = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
if size == 'all':
    X_train, y_train = agnews_tt['input'], agnews_tt['label']
else:

    # X_train, y_train = agnews['input'][:size], agnews['label'][:size]
    _, X_train, _, y_train = train_test_split(agnews_tt['input'], agnews_tt['label'], test_size=size/len(agnews_tt['input']), random_state=42)

X_test, X_dev, y_test, y_dev = train_test_split(agnews_all['text'], agnews_all['label'], test_size=0.5, random_state=42)

In [154]:
agnews_all['text'][90:100]

90    U.S. Brokers Cease-fire in Western Afghanistan...
91    Sneaky Credit Card Tactics Keep an eye on your...
92    Intel Delays Launch of Projection TV Chip In a...
93    Fund pessimism grows NEW YORK (CNN/Money) - Mo...
94    Kederis proclaims innocence Olympic champion K...
95    Eriksson doesn #39;t feel any extra pressure f...
96    Injured Heskey to miss England friendly NEWCAS...
97    Staples Profit Up, to Enter China Market  NEW ...
98    Delegation Is Delayed Before Reaching Najaf AG...
99    Consumer Prices Down, Industry Output Up  WASH...
Name: text, dtype: object

In [155]:
agnews_all['label'][90:100]

90    0
91    2
92    3
93    2
94    1
95    1
96    1
97    2
98    0
99    2
Name: label, dtype: int64

In [156]:
agnews_tt['input']

0      IBM to Quit Making PCs An era will come to an ...
1      Sun: 1.9 million downloads of Java fix (InfoWo...
2      Former Indonesian intelligence chief says emba...
3      RFK due to retain prime soccer seating DC Unit...
4      Creditors to forgive up to 80 per cent of Iraq...
                             ...                        
995    A Gandhi preaches peace in Mideast RAMALLAH, W...
996    National League Game Summary - Los Angeles at ...
997    Cardinals, Red Sox Use Homers to Win Baseball ...
998    BMC update aims to nip downtime in the bud Bat...
999    William surfs to his fathers rescue ONCE again...
Name: input, Length: 1000, dtype: object

In [157]:
X_train

521    Study: Hunters off hook for bison population c...
737    Griffin Gets Good News Th Redskins' defense re...
740    Protester attacks Berlin exhibition of art  #3...
660    Macromedia stretches Flex's features The compa...
411    Arrington Re-Injures Knee in Practice ASHBURN,...
                             ...                        
109    Sachs pays Ford \$13.7M to settle Investment b...
430    N.Y. Mayor Has Plans To Import Flu Shots Scram...
77     Windows XP SP2 respite to end After a nine-day...
84     Air Force  Pitch for  Boeing  Detailed E-mail ...
286    Prince Of Persia Returns Ubisoft has released ...
Name: input, Length: 250, dtype: object

In [158]:
X_train_processed = process_data(X_train)
X_dev_processed = process_data(X_dev)
X_test_processed = process_data(X_test)

In [159]:
agnews['input']

0      Fears for T N pension after talks Unions repre...
1      The Race is On: Second Private Team Sets Launc...
2      Ky. Company Wins Grant to Study Peptides (AP) ...
3      Prediction Unit Helps Forecast Wildfires (AP) ...
4      Calif. Aims to Limit Farm-Related Smog (AP) AP...
                             ...                        
495    Pressure points ATHENS -- The booing went on f...
496    Unions protest as overtime rules take effect W...
497    Serb denies siege terror charges A Bosnian Ser...
498    11th-hour highlights too late NBC's prime-time...
499    No IE? No Can See One thing that #39;s always ...
Name: input, Length: 500, dtype: object

In [160]:
X_train

521    Study: Hunters off hook for bison population c...
737    Griffin Gets Good News Th Redskins' defense re...
740    Protester attacks Berlin exhibition of art  #3...
660    Macromedia stretches Flex's features The compa...
411    Arrington Re-Injures Knee in Practice ASHBURN,...
                             ...                        
109    Sachs pays Ford \$13.7M to settle Investment b...
430    N.Y. Mayor Has Plans To Import Flu Shots Scram...
77     Windows XP SP2 respite to end After a nine-day...
84     Air Force  Pitch for  Boeing  Detailed E-mail ...
286    Prince Of Persia Returns Ubisoft has released ...
Name: input, Length: 250, dtype: object

In [161]:
cf_train = agnews_tt['counterfactual'][list(X_train.index)]
cf_train_y = agnews_tt['prediction'][list(X_train.index)]
# cf_dev = agnews['counterfactual'][list(X_dev.index)]
# cf_test = agnews['counterfactual'][list(X_test.index)]


In [162]:

data_for_vectorizer = X_train_processed.apply(lambda tokens: " ".join(tokens))
vectorizer = CountVectorizer(max_features=20000)
vectorizer.fit(data_for_vectorizer)
# Get the vocabulary mapping (word to integer index)
vocabulary = vectorizer.vocabulary_

# define a reusable lambda
to_indices = lambda texts: [token2index(text, vocabulary, max_length=max_length) for text in texts]

# now you can call it on any dataset
tfidf_train = to_indices(X_train_processed)
tfidf_dev   = to_indices(X_dev_processed)
tfidf_test  = to_indices(X_test_processed)

tfidf_train_cf = to_indices(process_data(cf_train))
# tfidf_dev_cf = to_indices(process_data(cf_dev))
# tfidf_test_cf = to_indices(process_data(cf_test))

In [163]:
y_train

521    3
737    1
740    0
660    3
411    1
      ..
109    2
430    2
77     3
84     2
286    3
Name: label, Length: 250, dtype: int64

In [164]:
cf_train_y

521    2
737    0
740    1
660    0
411    0
      ..
109    0
430    0
77     0
84     0
286    0
Name: prediction, Length: 250, dtype: int64

In [165]:


"""
########################################################################################################################
Save embeddings
########################################################################################################################
"""
# Stripping html from unprocessed text, just to clean it up
print('\ntrain_Data')
cf_train={'original': compile_ag(np.array(tfidf_train),y_train,X_train,tfidf_train_cf,cf_train)} # X,y,text,cf_X,cf_text

for key,val in cf_train['original']['K'].items():
    print(key,np.shape(val))
    print(val[0:10])

print('\ndev_Data')
cf_dev={'original':{ 
                    'text':X_dev,
                    'X':tfidf_dev,
                    'Y':y_dev,
                    'K':{'vector':None,'label':None,'magnitude':None, 'text':None}
                    }}

print('\ntest_Data')
cf_test={'original':{ 
                    'text':X_test,
                    'X':tfidf_test,
                    'Y':y_test,
                    'K':{'vector':None,'label':None,'magnitude':None, 'text':None}
                    }}

    
print('\ncontrol_Data')
flattened_text=[t for ts in cf_train['original']['K']['text'] for t in ts]
flattened_X = [x for xs in cf_train['original']['K']['X'] for x in xs]
flattened_Y = [y for ys in cf_train['original']['K']['Y'] for y in ys]


cf_control={'original': {
            'text':flattened_text,
            'X':np.array(flattened_X),
            'Y':list(cf_train_y),
            'K':{
                'text':cf_train['original']['text'],
                'X':np.array(np.zeros_like(flattened_X)),
                'Y':np.array(np.zeros_like(flattened_Y)),
                'K':np.array(np.zeros_like(flattened_X)),
                'magnitude':np.array(np.zeros_like(flattened_X)),
                }}}
    

pickle_data = {'train':cf_train,
               'test':cf_test,
               'dev':cf_dev,
               'control':cf_control,
               'n_classes':len(np.unique(y_test))}        


# pickle_data = {'train':cf_train,'test':cf_test,'dev':cf_dev,'n_classes':len(np.unique(y_test))}

embedding_path = f'data/tt_integer_len{max_length}_SIZE_{size}.pkl'
print(f"Saving to {embedding_path}")
with open(embedding_path, 'wb') as file:
    pickle.dump(pickle_data, file)



train_Data
Returning 250 samples with 250 counterfactuals
X (250, 1, 32)
[[[2178 1062 1045  242 1683  525  234  911 1062 1370 1045 1241 2349 1711
   2484 2334  770  242 1683 2006  525 2277 2530   48    0    0    0    0
      0    0    0    0]]

 [[ 970  933  953 1483 2263 1827  582 1820  953 1483  506  970 1441  754
   1029 1112  582 2218 1402 1668 2195  682    0    0    0    0    0    0
      0    0    0    0]]

 [[1754  156  227  762  135  281 1464  255  206 1969 2291  227  135 2020
     68  907 1464 1778 1426 1776 2505 1754 1579 2449 2403 2350 2511  762
   2534    0    0    0]]

 [[1328 2168  848  805  462  763 1991 2068  526 2053 1126 2469  114    0
      0    0    0    0    0    0    0    0    0    0    0    0    0    0
      0    0    0    0]]

 [[ 133 1839 1211 1705  139 2396 1245  133 2163 1886 1211 1705 1425 1995
   2281 1732  287 1284 1047 1668 2471 1413  882  911    0    0    0    0
      0    0    0    0]]

 [[ 820 1714 1765 2100  641 1790  957 1410 1260  329 2534 2172 193

/Users/jonathanerskine/University of Bristol/gradient_supervision/counterfactual-gradient-alignment-cli/cli_venv/lib/python3.10/site-packages/numpy/_core/numeric.py:442: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


creating baseline with counterfactuals

In [166]:
cf_train['original']['text'][0:2]

['Study: Hunters off hook for bison population crash Big game hunters may be off the hook in the latest twist of a prehistoric whodunit that tries to explain why bison populations sharply crashed thousands of years ago.',
 "Griffin Gets Good News Th Redskins' defense receives good news on Cornelius Griffin's MRI exam of a hip injury, and the defensive tackle might play Sunday against the Eagles."]

In [167]:
print(np.unique(cf_control['original']['Y']))

[0 1 2 3]


   
        "world": 0,
        "sports": 1,
        "business": 2,
        "science/technology": 3
    

In [168]:
for i in range(20,25):
    print("Original: ",cf_train['original']['Y'][i],cf_train['original']['text'][i])
    print("Counter: ",cf_control['original']['Y'][i],cf_control['original']['text'][i])

Original:  0 Blair Won't Apologize Over Iraq LONDON - Prime Minister Tony Blair denied Wednesday that he misrepresented intelligence about Iraqi weapons before the war, rejecting growing demands in Parliament that he apologize.   "I cannot bring myself to say that I misrepresented the evidence, since I do not accept that I did," Blair said in the House of Commons...
Counter:  1 Serena Williams Won't Apologize Over US Open Outburst  NEW YORK - Tennis star Serena Williams denied Wednesday that she overreacted during her US Open final loss, rejecting growing criticism from fans and officials that she apologize. "I cannot bring myself to say that I overreacted, since I do not accept that I did," Williams said in a post-match press conference...
Original:  3 New Computer Associates CEO to Get Bonus (AP) AP - Computer Associates International Inc., a business software company embroiled in an accounting scandal, is giving its new chief executive, John Swainson, a pay package that totals more 

In [169]:
total = 0
for y,y_p in list(zip(cf_train['original']['Y'],cf_control['original']['Y'])):
    
    if y != y_p:
        total+=1

print(total/len(cf_control['original']['Y']))


1.0
